In [1]:
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from RLAlg.utils import set_seed_everywhere

from model.encoder import FrameObservationEncoderNet, EncoderNet
from state_frame_dataset import get_dataloader

In [2]:
class Alignment(nn.Module):
    def __init__(self, vector_weight):
        super().__init__()
        
        self.frame_encoder = FrameObservationEncoderNet(6, 256)

        encoder = EncoderNet(39, [256, 256])

        encoder.load_state_dict(vector_weight)

        self.vector_encoder = encoder.layers[:]

        for param in self.vector_encoder.parameters():
            param.requires_grad = False

    def encoder_vectors(self, vectors):
        vector_features = vectors
        for layer in self.vector_encoder:
            vector_features = layer(vector_features)

        return vector_features

    def encode_frames(self, frames):
        frame_features = self.frame_encoder(frames)

        return frame_features
    
    def forward(self, frames, vectors):
        frame_features = self.encode_frames(frames)
        vector_features = self.encoder_vectors(vectors)

        return frame_features, vector_features

In [3]:
weight_folder = f"weights/ddpg/push"
        
if not os.path.exists(weight_folder):
    os.makedirs(weight_folder)

set_seed_everywhere(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder_weight, _, _ = torch.load(f"{weight_folder}/actor_0_100.pt", weights_only=True)
model = Alignment(encoder_weight).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=10)
dataloader = get_dataloader("push")
size = len(dataloader.dataset)

In [ ]:
def contrastive_loss(q, k):
    return (1-F.cosine_similarity(q, k, dim=-1)).mean()
    #return F.mse_loss(q, k)

In [5]:
for _ in range(10):
    start_time = time.time()
    running_loss = 0.0
    for _, (vectors, frames) in enumerate(dataloader):
        vectors = vectors.to(device)
        frames = frames.to(device)
        
        frame_features, vector_features = model(frames, vectors)
        
        loss = contrastive_loss(frame_features, vector_features)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * vectors.size(0)
    
    scheduler.step()
    end_time = time.time()
    train_loss = running_loss / size
    print(f"Train Loss: {train_loss:.4f}")
    elapsed_time = end_time - start_time
    print(f"The loop took {elapsed_time:.2f} seconds to complete.")

torch.save(model.frame_encoder.state_dict(), f"weights/ddpg/push/frame_encoder_0_mse.pt")

Train Loss: 0.0633
The loop took 31.03 seconds to complete.
Train Loss: 0.0341
The loop took 30.99 seconds to complete.
Train Loss: 0.0303
The loop took 31.07 seconds to complete.
Train Loss: 0.0277
The loop took 31.10 seconds to complete.
Train Loss: 0.0249
The loop took 31.09 seconds to complete.
Train Loss: 0.0219
The loop took 30.93 seconds to complete.
Train Loss: 0.0185
The loop took 30.75 seconds to complete.
Train Loss: 0.0148
The loop took 30.84 seconds to complete.
Train Loss: 0.0112
The loop took 31.03 seconds to complete.
Train Loss: 0.0086
The loop took 30.82 seconds to complete.
